# 04 · PlantSeg field leaves (upgrade, Colab T4)

Runs the seed 42 ResNet-50 from notebook 01 over real field photographs of grape leaves from
PlantSeg v6 (Zenodo record 16665770, CC BY 4.0; DECISIONS.md D14). It uses only the leaves
selected in `results/plantseg/leaf_filter.csv` (D15), each photograph once. Two analyses,
never merged (D16):

* **Black rot leaves (n = 25):** the one PlantSeg class the model knows. Accuracy with a
  Wilson interval, mean confidence, ECE, the confidence distribution. n is small.
* **Downy mildew, leaf spot, leafroll leaves:** none of the model's classes is correct, so
  there is **no accuracy**: out-of-distribution confidence, and which class it assigns.

Both are compared with the same model on clean PlantVillage test images and on the most
severe simulated corruption, and with the threshold chosen on validation.

**Needs:** notebooks 01 and 02 finished (the seed 42 run and its sweep in `OUT`). It does not
need the PlantVillage zip. It downloads PlantSeg (1.7 GB) into this Colab session, not to
Drive, and checks its MD5. **No PlantSeg image is written to Drive**: only numbers, logits and
figures.

In [ ]:
# ---- settings: the only cell you normally edit ---------------------------------
REPO_URL = "https://github.com/brycedaniliuc/COMP2035-grape-calibration.git"
REPO_REF = "main"                       # a branch, tag or commit hash
ZIP_PATH = "/content/drive/MyDrive/grape_dataset_for_colab.zip"   # must match Drive
DATA_ROOT = "/content/grape_dataset"    # the zip extracts to this folder
OUT = "/content/drive/MyDrive/COMP2035_runs"   # every output is written here, as it goes
PLANTSEG_URL = "https://zenodo.org/api/records/16665770/files/Plantseg.zip/content"
PLANTSEG_ZIP = "/content/Plantseg.zip"   # this session only, never Drive

In [ ]:
import os, subprocess, sys, time
from pathlib import Path
T0 = time.time()

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")

REPO = Path("/content/COMP2035-grape-calibration")
if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "--tags"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_REF], check=True)
commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], check=True,
                        capture_output=True, text=True).stdout.strip()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("code at", REPO_REF, commit)

# Colab already has torch, torchvision, numpy, scikit-learn, scikit-image and matplotlib.
# Do NOT pip install requirements.txt here: it pins the CPU build of torch.
import importlib.util
missing = [m for m in ("skimage", "sklearn", "nbformat") if importlib.util.find_spec(m) is None]
if missing:
    pkg = {"skimage": "scikit-image", "sklearn": "scikit-learn"}
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *[pkg.get(m, m) for m in missing]], check=True)

from src.train import environment, select_device
import json
env = environment(select_device())
print(json.dumps(env, indent=1))
if env["device"] != "cuda":
    print("!! NO GPU: Runtime > Change runtime type > T4 GPU, then run all again")
os.makedirs(OUT, exist_ok=True)
print("outputs ->", OUT)

## The seed 42 run (notebook 01) and its sweep (notebook 02)

In [ ]:
from src.ensemble import find_run
BASE = find_run(OUT, seed=42)
assert BASE, f"no finished resnet50 source run in {OUT}: run notebook 01 first"
RUN_ID = json.loads(BASE.read_text())["run_id"]
for f in (f"sweep_{RUN_ID}.json", f"sweep_logits_{RUN_ID}.npz"):
    assert (BASE.parent / f).exists(), f"{f} missing: run notebook 02 first"
print("model:", BASE.name)

## PlantSeg v6 from Zenodo (about 1–2 min), checked against its MD5

In [ ]:
import shutil, urllib.request
from src.plantseg import SOURCE, md5
if not (os.path.exists(PLANTSEG_ZIP) and md5(PLANTSEG_ZIP) == SOURCE["md5"]):
    with urllib.request.urlopen(PLANTSEG_URL) as r, open(PLANTSEG_ZIP, "wb") as fh:
        shutil.copyfileobj(r, fh, length=1 << 22)
got = md5(PLANTSEG_ZIP)
assert got == SOURCE["md5"], f"download is not PlantSeg v6 (md5 {got}): run the cell again"
print(f"PlantSeg v6 OK: {os.path.getsize(PLANTSEG_ZIP) / 1e9:.2f} GB, md5 {got}")

## The two analyses (about 1 min)

In [ ]:
from src.plantseg import plantseg_eval
PS = plantseg_eval(BASE, PLANTSEG_ZIP)
PS_PATH = f"{OUT}/plantseg_{RUN_ID}.json"

## Figures (paper PDF + slide PNG), to Drive

In [ ]:
from src.figures import render_plantseg
paths = render_plantseg(PS_PATH, f"{OUT}/figures_{RUN_ID}", tag=RUN_ID)
print(len(paths), "files ->", f"{OUT}/figures_{RUN_ID}")

## Verify (Tier 1 must pass)

In [ ]:
r = subprocess.run([sys.executable, "-m", "src.verify", "--plantseg", PS_PATH],
                   capture_output=True, text=True)
print(r.stdout[-6000:], r.stderr[-3000:])
assert r.returncode == 0, "a Tier 1 check failed: stop and investigate"
print(f"total elapsed {(time.time() - T0) / 60:.1f} min")

## Everything this notebook wrote to Drive

In [ ]:
for p in sorted(Path(OUT).rglob("*plantseg*")):
    if p.is_file():
        print(f"{p.stat().st_size / 1e6:9.2f} MB  {p.relative_to(OUT)}")